# Whisper 학습 노트북

이 노트북은 OpenAI Whisper 모델을 한국어 GOLD JSONL 로 파인튜닝하기 위한 셀 골격입니다.

## 사용 흐름

1. 환경 점검 (셀 1~3)
2. 데이터 로드 + 검증 (셀 4~5)
3. Whisper Dataset 어댑터 (셀 6)
4. 학습 + 저장 (셀 7~8) — HuggingFace Trainer 노트북 내 직접 실행

평가는 이 노트북 밖에서 한다 (BENCHMARK/configs/*.yaml — 맨 아래 안내 참고).

## 약속

- 핵심 로직은 `project/` 모듈에. 노트북은 *호출만*.
- 모든 경로/백본/W&B 는 `configs/<exp>.yaml` 에서 읽는다.

## 셀 1 — 환경 점검

In [ ]:
import sys, os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "2"   # ★ 빈 GPU 번호! torch import 前 필수 (GPU 0/1 풀이면 OOM)
from pathlib import Path

REPO = Path('/home/cssong/workspace/TRAIN-ASR')
sys.path.insert(0, str(REPO))
os.chdir(REPO)

print('CONDA_DEFAULT_ENV:', os.environ.get('CONDA_DEFAULT_ENV'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)'))
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total', '--format=csv'], check=False)

## 셀 2 — Config 로드

In [ ]:
from project.utils.config import load_config
from project.utils.seed import seed_everything

CFG_PATH = REPO / 'configs/default.yaml'
cfg = load_config(CFG_PATH)
seed_everything(cfg['experiment']['seed'])
print('experiment:', cfg['experiment']['name'])
print('backbone:  ', cfg['models']['whisper']['backbone'])
print('train data:', cfg['paths']['train_jsonl'])

## 셀 3 — 의존성 sanity

In [ ]:
import importlib

needed = ['transformers', 'datasets', 'accelerate', 'evaluate', 'jiwer', 'soundfile', 'torch']
for pkg in needed:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'  {pkg:14s} {v}')
    except ImportError:
        print(f'  {pkg:14s} ❌ 미설치')

import torch
print('CUDA available:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())

GOLD JSONL 로드 + 자동 검증. 실패 시 `SchemaValidationError`.

In [ ]:
from project.data import load_samples

train = load_samples(cfg['paths']['train_jsonl'])
val   = load_samples(cfg['paths']['val_jsonl'])
print(f'train: {len(train):,}  /  val: {len(val):,}')
print('sample[0]:', train[0])

## 셀 5 — 빠른 분포 점검 (SV 노트북과 동일)

In [ ]:
from collections import Counter

# 새 스키마: 필수 5 + 선택(age/gender). speaker_id/corpus_id/labeling 은 제거됨.
# 코퍼스 구분이 필요하면 key 접두사로: s.key.split("__")[0]
print(f'gender: {Counter(s.gender for s in train).most_common()}')
print(f'age:    {Counter(s.age for s in train).most_common()}')
print(f'corpus(키 접두사): {Counter(s.key.split("__")[0] for s in train).most_common()}')

## 셀 6 — Whisper Dataset 어댑터 (Sample → HuggingFace Dataset)

`project/data/adapters/whisper.py` 의 `to_whisper_dataset` 가 오디오→로그멜(input_features), 텍스트→토큰(labels) 로 변환한다.

In [ ]:
# GOLD Sample → HuggingFace Dataset (input_features + labels)
from project.data.adapters.whisper import to_whisper_dataset

m = cfg['models']['whisper']
_kw = dict(
    backbone=m['backbone'],
    language=m.get('language', 'ko'),
    task=m.get('task', 'transcribe'),
    text_field=cfg['data']['text_field'],
    sampling_rate=cfg['data']['sample_rate'],
)
train_ds = to_whisper_dataset(train, **_kw)
val_ds   = to_whisper_dataset(val,   **_kw)
print(train_ds)
print(val_ds)

## 셀 7 — 모델 / Trainer 빌드 (HF Trainer)

In [ ]:
# HF Seq2SeqTrainer 빌드 (CER 모니터링·cosine 스케줄러·early stopping 포함)
from project.training.whisper import build_trainer

trainer = build_trainer(cfg, train_dataset=train_ds, eval_dataset=val_ds)
print('output_dir   :', trainer.args.output_dir)
print('lr_scheduler :', trainer.args.lr_scheduler_type)
print('best metric  :', trainer.args.metric_for_best_model, '(낮을수록 좋음)')

## 셀 8 — 학습 실행

노트북 셀 내에서 실행. 단, *장시간 학습은 tmux/screen* 외부 실행 권장.

In [ ]:
# 학습 + 저장. 장시간이면 tmux 권장 (노트북 커널이 꺼지면 학습도 죽음).
from pathlib import Path

trainer.train()   # 재개: trainer.train(resume_from_checkpoint='<checkpoint dir>')

output_dir = Path(cfg['paths']['outputs_dir']) / cfg['experiment']['name']
trainer.save_model(str(output_dir))
(output_dir / 'config.yaml').write_text(
    __import__('yaml').safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('saved →', output_dir)

# ── 위 4셀(어댑터→빌드→학습→저장)의 한 줄 등가물 (CLI 와 동일 경로) ──
#   from project.training.run import run_whisper_training
#   run_whisper_training(cfg)
# 셸:  python scripts/train.py --config configs/<exp>.yaml --model whisper

# ── 학습 후 GPU 메모리 해제 (커널이 계속 점유하지 않도록) ──
import gc, torch
del trainer
gc.collect(); torch.cuda.empty_cache()
print('GPU 메모리 해제 (완전 해제는 Kernel Restart)')


## 평가는 별도 — BENCHMARK 에서

이 노트북은 **학습까지**만 담당한다. 학습된 체크포인트(`outputs/<exp>/`) 평가는
BENCHMARK 쪽에서 일원화한다 (학습/평가 config 분리):

- 평가 설정: `BENCHMARK/configs/*.yaml` 의 `recognizer.model_path` 를
  `outputs/<exp>/` 로 지정 → `benchmarks` 리스트에 평가셋 추가
- 실행: `notebooks/00_build_benchmark.ipynb` / `BENCHMARK/README.md §3`